# wf_v4.4 real runs with Llama (diary Nov 14): last tuning before the freeze
Code: GitHub `TriDang2025/attribute-extraction`, branch `wf-v4.4` (from `wf-v4.3`), folder `step1a/`. Results in `results/wf_v44/`.

Model: Llama-3.1-8B-Instruct, bf16, vLLM, temperature 0, guided JSON from each job's schema (setting `llama8b`). Runtime: **A100 (40 GB or more)**.
Every call row keeps the guided raw text (also when the unguided retry wins) and its own seconds; the model loading time goes to `results/wf_v44/model_load.json`.

- Cells 1-5: setup (as the wf_v4.3 notebook; cell 4 also logs the loading time).
- Cell 6: data check.
- Cell 7: **T1** guided names probe: 3 jobs per schema variant, picks the first that parses 3/3 -> `t1/choice.json`. About 15 calls, 2-5 min (estimate).
- Cell 8: **T2** keyed split pass on the 54 v4.3 split jobs (target at most 5/54 parse errors). About 54 calls, about 5 min (estimate).
- Cell 9: **T4** Step 3 v4.4 (similar batches, recall rescue, duplicate merge, then split -> reconcile -> dupcheck), L2 score, writes `a_llama44`. About 400-500 calls, 25-35 min (estimate).
- Cell 10: **T6** Step 4 v4.4 on a_llama44 then c1 (dev50 + dev50b, seeds 0-2 + vote), score. 30-40 min (estimate).
- Cell 11: **T7** Step 4 on/off checks (dev50, seed 0, one change off at a time). 15-20 min (estimate).
- Cell 12: Step 3 on/off checks (each booster off, order 0). 15-25 min (estimate).
- Each cell commits and pushes its results folder. If time runs short, the order of priority is cells 7, 8, 9, 10, 11, 12.
- After a crash or disconnect: rerun cells 1, 3, 4, 5 and then the cell you were on. Answered calls are read from the cache.
- Colab Secrets `HF_TOKEN` and `GH_TOKEN`, notebook access on. Never put a token in a cell.
- When the GPU work is done: **Runtime > Disconnect and delete runtime**, so the paid GPU stops.


In [ ]:
#@title 1. Tokens, clone/pull wf-v4.4, git identity
GIT_NAME = "Your Name"  #@param {type:"string"}
GIT_EMAIL = "you@users.noreply.github.com"  #@param {type:"string"}
BRANCH = "wf-v4.4"  #@param {type:"string"}
import os, subprocess
from google.colab import userdata
from huggingface_hub import login

hf_token = (userdata.get("HF_TOKEN") or "").strip()
if hf_token:
    login(token=hf_token)
del hf_token
os.environ["GH_TOKEN"] = (userdata.get("GH_TOKEN") or "").strip()
def run(cmd, **kw): subprocess.run(cmd, check=True, **kw)
run(["git", "config", "--global", "user.name", GIT_NAME])
run(["git", "config", "--global", "user.email", GIT_EMAIL])
run(["git", "config", "--global", "credential.helper",
     '!f() { echo username=token; echo "password=$GH_TOKEN"; }; f'])
REPO = "/content/attribute-extraction"
if not os.path.exists(REPO):
    run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/TriDang2025/attribute-extraction.git", REPO])
else:
    run(["git", "-C", REPO, "checkout", "-q", BRANCH])
    run(["git", "-C", REPO, "pull", "-q", "--rebase", "--autostash"])
%cd /content/attribute-extraction/step1a
!git log -1 --format="code at %h (%an, %ar): %s"

**Cell 2 note:** the install prints dependency-resolver warnings; ignore them. Then **Runtime > Restart session**, rerun cell 1 and skip cell 2.
gliner and sentence-transformers are installed with `--no-deps` so they do not replace the torch / transformers that vLLM needs. Cell 4 checks that they load.

In [ ]:
#@title 2. Install (pinned; a few minutes; then restart the session and skip this cell)
!pip install -q vllm==0.33.0
!pip install -q --no-deps gliner==0.2.29 sentence-transformers==5.6.0
!pip install -q sentencepiece

In [ ]:
#@title 3. Model setting (config/wf_v41_models.json) and GPU check
SETTING = "llama8b"  #@param ["llama8b"]
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a")
from s1a import nov16_v41 as NV
SET = NV.model_setting(SETTING)
print(SETTING, SET)
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "no GPU: change the runtime type to A100"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU memory {gb:.1f} GiB")
assert gb >= SET["min_gpu_gb"], f"only {gb:.1f} GiB: {SETTING} needs {SET['min_gpu_gb']} GiB (an A100)"

In [ ]:
#@title 4. Load the model (gpu_memory_utilization leaves room for bge-m3 of T4)
MODEL, REVISION = SET["model"], SET["revision"]   # from the setting of cell 3
GPU_UTIL = 0.80  #@param {type:"number"}
MAX_LEN = 32768
import vllm
from vllm import LLM, SamplingParams
print("vllm", vllm.__version__, "| model", MODEL, "| revision", REVISION)
import time
_t0 = time.time()
llm = LLM(model=MODEL, revision=REVISION, tokenizer_revision=REVISION, quantization=SET["quantization"],
          max_model_len=MAX_LEN, gpu_memory_utilization=GPU_UTIL)
from s1a import log_v44 as L44
print("model load:", L44.save_load_time("results/wf_v44/model_load.json", time.time() - _t0, MODEL, REVISION))


In [ ]:
#@title 5. Backend: temperature 0, guided JSON per job schema, batches, cache per job (s1a/log_v44.VllmBackend44)
BATCH = 64  #@param {type:"integer"}
def sampling(max_tokens, schema):
    kw = dict(temperature=0, max_tokens=max_tokens)
    if schema is not None:
        try:
            from vllm.sampling_params import StructuredOutputsParams
            kw["structured_outputs"] = StructuredOutputsParams(json=schema)
        except ImportError:
            from vllm.sampling_params import GuidedDecodingParams
            kw["guided_decoding"] = GuidedDecodingParams(json=schema)
    return SamplingParams(**kw)
from s1a import log_v44 as L44
backend = L44.VllmBackend44(llm, MODEL, REVISION, sampling, max_len=MAX_LEN, batch=BATCH, vllm_version=vllm.__version__)
backend.model_id

In [ ]:
#@title 6. Data check: bge-m3 vectors, GT levels, frozen prompts, v4.3 split jobs
from pathlib import Path
need = ["results/graph_v22_dev50/spans/vectors.npz", "results/graph_v22_dev50b/spans/vectors.npz",
        "results/loop1_v35/gt_levels.csv", "results/loop1_v37/dev50b/gt/gt_levels.csv",
        "results/wf_v42/t1b_step3_keyed/manifest.json", "results/wf_v43/u2/manifest.json",
        "config/wf_v44_step3.json", "config/wf_v44_step4.json"]
missing = [p for p in need if not Path(p).exists()]
print("missing:", missing or "none")
assert not missing, "files missing: stop and tell Claude"


In [ ]:
#@title 7. T1: guided names probe (variants v1, v2, v3, v4, v0; 3 jobs each) -> results/wf_v44/t1/choice.json
import subprocess
from s1a import step3_v44 as S3
choice, info = S3.run_probe(S3.OUT_T1, backend)
print("choice:", choice)
print(info)
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v44 T1 names probe", "wf_v44/t1"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
print("T1 done")


In [ ]:
#@title 8. T2: keyed split pass on the 54 v4.3 split jobs (target <= 5/54 parse errors)
import sys, subprocess
sys.path.insert(0, "/content/attribute-extraction/step1a/scripts")
import t2_split_v44 as T2
T2.build(T2.OUT)
print(T2.run(T2.OUT, backend))
print(T2.report(T2.OUT))
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v44 T2 keyed split", "wf_v44/t2"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
print("T2 done")


In [ ]:
#@title 9. T4: Step 3 v4.4 (boosters on) -> L2 score -> a_llama44
import subprocess
from s1a import step3_v44 as S3
encoder = S3.bge_m3_encoder()
info, sizes = S3.full_run(S3.OUT_T4, backend, S3.load_config(), encoder, name_vecs=S3.OUT_T4 / "name_vecs.npz", save_vecs=True)
print(info)
print("a_llama44:", sizes)
for f in ("summary.md", "calls_v44.json", "booster_stats.json"):
    p = S3.OUT_T4 / "scores" / f
    if p.exists():
        print("==", f); print(p.read_text(encoding="utf-8")[:3000])
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v44 T4 step3 v44", "wf_v44/t4"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
print("T4 done")


In [ ]:
#@title 10. T6: Step 4 v4.4 on a_llama44, then c1 (dev50 + dev50b, seeds 0-2 + vote), then score
import sys, subprocess
sys.path.insert(0, "/content/attribute-extraction/step1a/scripts")
import t6_step4_v44 as T6
for inp in ("a_llama44", "c1"):                  # a_llama44 needs cell 9
    print(inp, T6.run(backend, inputs=(inp,)))
    p = subprocess.run(["bash", "colab/sync.sh", "push-results", f"wf_v44 T6 step4 v44 {inp}", "wf_v44/t6"], capture_output=True, text=True)
    print(p.stdout.strip() or p.stderr[-500:])
T6.score()
print((T6.OUT / "summary.md").read_text(encoding="utf-8")[:4000])
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v44 T6 scores", "wf_v44/t6"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
print("T6 done")


In [ ]:
#@title 11. T7: Step 4 on/off checks (dev50, seed 0, input a_llama44; all-on reused from T6)
import sys, subprocess
sys.path.insert(0, "/content/attribute-extraction/step1a/scripts")
import t6_step4_v44 as T6
import t7_ablation_v44 as T7
print(T7.run(backend, all_on_dir=T6.OUT))
print((T7.OUT / "t7_table.md").read_text(encoding="utf-8"))
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v44 T7 step4 on/off", "wf_v44/t7"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
print("T7 done")


In [ ]:
#@title 12. Step 3 on/off checks (each booster off, order 0; reads T4's name vectors)
import subprocess
from s1a import step3_v44 as S3
for sw in ("similar_batches", "rescue", "merge"):
    print(sw, S3.ablation(sw, backend, S3.load_config(), encoder))
    p = subprocess.run(["bash", "colab/sync.sh", "push-results", f"wf_v44 step3 off {sw}", "wf_v44/t4_ablate"], capture_output=True, text=True)
    print(p.stdout.strip() or p.stderr[-500:])
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v44 model load time", "wf_v44"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
print("all done: disconnect the runtime, then tell Claude 'wf_v4.4 run done'")
